In [2]:
import os
import re
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from tbparse import SummaryReader
from tueplots import bundles
from tueplots.constants.color import palettes

# Keep plotting style consistent with existing slides
plt.rcParams.update(bundles.beamer_moml())
plt.rcParams.update({'font.sans-serif': 'DejaVu Sans', 'figure.dpi': 200})

# change directory to project root
os.chdir(os.path.expanduser("~/Desktop/pomdp_coder"))
print(os.getcwd())
base_dir = os.path.join("outputs")

C:\Users\Frederik\Desktop\pomdp_coder


In [3]:
def load_avg_reward_df(path: str):
    """Return the Average Episode Reward scalars for a given experiment directory."""
    reader = SummaryReader(path, extra_columns={'dir_name'})
    df = reader.scalars
    return df[df['tag'] == "Episode Reward"].reset_index(drop=True)
    # return df[df['tag'] == "Average Episode Reward"].reset_index(drop=True)

def get_clean_df_qwen3(path: str):
    """Return a cleaned DataFrame with only relevant columns."""
    df = load_avg_reward_df(path)
    df['seed'] = df['dir_name'].apply(lambda x: re.search(r"_seed(\d+)", x).group(1))
    df['environment'] = path.split("\\")[-3]
    df['approach'] = path.split("\\")[-2]
    df['episode'] = df['step'] 
    df = df.drop(columns=['tag', 'step', 'dir_name'])
    df = df[['environment', 'approach', 'seed', 'episode', 'value']]
    return df

In [4]:
# add "four_rooms", later, not finished yet
envs = ["tiger", "rocksample", "empty", "corners", "lava", "four_rooms", "unlock"]
method = "ours"

directories = {
    env: os.path.join(base_dir, env, "ours", "2025-12-04")
        for env in envs
}
directories

{'tiger': 'outputs\\tiger\\ours\\2025-12-04',
 'rocksample': 'outputs\\rocksample\\ours\\2025-12-04',
 'empty': 'outputs\\empty\\ours\\2025-12-04',
 'corners': 'outputs\\corners\\ours\\2025-12-04',
 'lava': 'outputs\\lava\\ours\\2025-12-04',
 'four_rooms': 'outputs\\four_rooms\\ours\\2025-12-04',
 'unlock': 'outputs\\unlock\\ours\\2025-12-04'}

In [5]:
dfs = []

for env, dir_path in directories.items():
    df = get_clean_df_qwen3(dir_path)
    dfs.append(df)

final_df = pd.concat(dfs, ignore_index=True)
final_df

,environment,approach,seed,episode,value
0,tiger,ours,0,0,0.903921
1,tiger,ours,0,1,0.922368
2,tiger,ours,0,2,0.941192
3,tiger,ours,0,3,0.903921
4,tiger,ours,0,4,0.903921
...,...,...,...,...,...
695,unlock,ours,1,5,0.709322
696,unlock,ours,1,6,0.784717
697,unlock,ours,1,7,0.753642
698,unlock,ours,1,8,0.817073


In [9]:
df_prev = pd.read_csv(os.path.join(base_dir, "process_outputs", "baseline_results.csv"))
df_prev = df_prev[df_prev['approach'] != "ours"]
df_prev

,environment,approach,seed,episode,value
0,tiger,hardcoded,0,0,0.941192
1,tiger,hardcoded,0,1,0.941192
2,tiger,hardcoded,0,2,0.941192
3,tiger,hardcoded,0,3,0.903921
4,tiger,hardcoded,0,4,0.941192
...,...,...,...,...,...
2785,four_rooms,random,9,5,0.000000
2786,four_rooms,random,9,6,0.000000
2787,four_rooms,random,9,7,0.000000
2788,four_rooms,random,9,8,0.000000


In [10]:
df = pd.concat([df_prev, final_df], ignore_index=True)
df

,environment,approach,seed,episode,value
0,tiger,hardcoded,0,0,0.941192
1,tiger,hardcoded,0,1,0.941192
2,tiger,hardcoded,0,2,0.941192
3,tiger,hardcoded,0,3,0.903921
4,tiger,hardcoded,0,4,0.941192
...,...,...,...,...,...
2795,unlock,ours,1,5,0.709322
2796,unlock,ours,1,6,0.784717
2797,unlock,ours,1,7,0.753642
2798,unlock,ours,1,8,0.817073


In [11]:
# write to csv
df.to_csv(os.path.join(base_dir, "process_outputs", "baseline_results_qwen3.csv"), index=False)